# PEYNNEY'S SOLUTION WITH AN EXTERNAL ELECTRIC FIELD IN ER

The dilaton-equivalent form of the action in ER reads:

\begin{eqnarray}
S
&=& \frac{1}{\tilde{\kappa}}\left(
\frac{\vartheta^{2} R}{2\tilde{\kappa}}
+ \vartheta \mathcal{L}_{m}\right).
\end{eqnarray}

The corresponding field equations are:

\begin{eqnarray}
\nabla_{\mu}\!\left(\vartheta F^{\mu\nu}\right) &=& 0, \\
3\vartheta^{-2}\Box\vartheta^{2}&=&\vartheta^{-1}\left(T-\mathcal{L}_{m}\right), \\
G_{\alpha\beta}
&=& \frac{T_{\alpha\beta}}{\vartheta}
+ \vartheta^{-2}\bigl(\nabla_{\alpha}\nabla_{\beta}
- g_{\alpha\beta}\Box \bigr)\vartheta^{2},
\end{eqnarray}

whith $\vartheta$ a scalar defined upon the ratio between the scalars $\mathcal{L}_{m}$ and $R$

\begin{equation}
    \vartheta := -\frac{\mathcal{L}_{m}}{R},
\end{equation}

and the stress-energy tensor 

\begin{equation}
T_{\mu\nu}=2\left(F_{\rho\mu}F^{\rho}_{\ \nu} - \frac{1}{4}g_{\mu\nu}F^{2}\right).
\end{equation}


We refer the reader to the notebook EINSTEIN_Penney_Melvin_Electric for a complete description of the Penney solution. Note that to run this notebbok and in particular to verify the field equations in ER, you need to have the python files expr_walkers.py and sage_fun.py stored in the same folder as this notebook. To reduce the computation time, one may also directly use the stored analytical expressions for the most computationally demanding quantities by placing the files ER_Penney_Melvin_dict.sobj and ER_Penney_Melvin_test_dict.sobj in the same directory and loading the corresponding expressions.

In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
%display latex

In [20]:
version()

'SageMath version 10.1, Release Date: 2023-08-20'

'SageMath version 10.1, Release Date: 2023-08-20'

In [21]:
from sage.manifolds.operators import dalembertian
from sage.manifolds.operators import laplacian
from sage.manifolds.operators import grad
from sage.symbolic.operators import FDerivativeOperator
from sage.symbolic.expression_conversions import ExpressionTreeWalker
from expr_walkers import explore_expression, check_res_expr, decomp_exp
from expr_walkers import subs_func
import sage_func
import importlib
import sage.symbolic.expression_conversions as ec
importlib.reload(ec)

<module 'sage.symbolic.expression_conversions' from '/home/mwavasseur/Software/Sage/src/sage/symbolic/expression_conversions.py'>

In [22]:
functions_list=['Omega','Lambda','Av','Ev']
def subs_func(arg):
    subs_funcs = [(Omega, Omega_expr),(Lambda,Lambda_expr),(Av,Av_expr),(Ev,Ev_expr)] 
    if hasattr(arg, 'expr'):
        arg = arg.expr()
    if hasattr(arg, 'apply_map')*hasattr(arg, 'display'):
        for i, (old_func, new_func) in enumerate(subs_funcs):
            arg.apply_map(lambda f: f.substitute_function(old_func, new_func).simplify_full())
            print('{0} substituted'.format(functions_list[i])) 
    elif hasattr(arg, 'apply_map'):
        return arg.apply_map(lambda f: subs_func(f).simplify_full())
    else:
        for i, (old_func, new_func) in enumerate(subs_funcs):
            arg = arg.substitute_function(old_func, new_func).simplify_full()
            print('{0} substituted'.format(functions_list[i]))
    print('Full Substitution : Done')
    return arg

In [23]:
M = Manifold(4, 'M', structure='Lorentzian')
print(M)

4-dimensional Lorentzian manifold M


In [24]:
XY.<t,r,th,ph> = M.chart(r"t r:(0,+oo) th:(0,pi):\theta ph:(0,2*pi):\varphi")
XY

Chart (M, (t, r, th, ph))

# I. Definition of the metric

In [25]:
m, xi, a, B = var('m xi alpha B')
assume(m > 0, 'real')
assume(xi > 0, 'real')
alpha = 1/(2*sqrt(3))

In [26]:
Omega = function('Omega')
Lambda = function('Lambda')
Av = function('A_v')
Ev = function('E_v')

In [27]:
g = M.metric()

g[0,0] = - Av(r) * Lambda(r,th)**(2/(1+alpha**2))*Omega(r,th)**2
g[1,1] = Ev(r,th) / Av(r) * Lambda(r,th)**(2/(1+alpha**2))*Omega(r,th)**2
g[2,2] = r**2 * Ev(r,th) * Lambda(r,th)**(2/(1+alpha**2))*Omega(r,th)**2
g[3,3] =  (r*sin(th))**2 / Lambda(r,th)**(2/(1+alpha**2))*Omega(r,th)**2

show(g.display())

g = -A_v(r)*Lambda(r, th)^(24/13)*Omega(r, th)^2 dt⊗dt + E_v(r, th)*Lambda(r, th)^(24/13)*Omega(r, th)^2/A_v(r) dr⊗dr + r^2*E_v(r, th)*Lambda(r, th)^(24/13)*Omega(r, th)^2 dth⊗dth + r^2*Omega(r, th)^2*sin(th)^2/Lambda(r, th)^(24/13) dph⊗dph

In [28]:
nab = g.connection(name=r'\nabla') 

# II. The vector potential

In [29]:
pot_vec = M.tensor_field(0,1,name='A')
pot_vec[0]=-(1/(2*sqrt(6)))*(m^2*cos(th)^2-2*m*r+r^2)^(-1/2)*(B*m^2-B*m*r)*cos(th)*xi-B*r*cos(th)+2*B*m*cos(th)
pot_vec[1]=0
pot_vec[2]=0
pot_vec[3]=0

show(pot_vec.display())

A = (2*B*m*cos(th) - B*r*cos(th) - 1/12*sqrt(6)*(B*m^2 - B*m*r)*xi*cos(th)/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)) dt

In [30]:
DF = nab(pot_vec) ; DF

Tensor field \nabla(A) of type (0,2) on the 4-dimensional Lorentzian manifold M

# III. Variables and functions

In [31]:
Av_expr(r) = (1- 2*m /r)
Ev_expr(r,th) = exp(- (xi**2 * m**2 * r**2 * Av_expr(r) * sin(th)**2) / (2 * (r**2*Av_expr(r) + m**2 * cos(th)**2)**2))
old_Phi = M.scalar_field({XY:m*xi / sqrt(2*(r**2*Av_expr(r) + m**2 * cos(th)**2))}, name=r'\varphi')
old_Psi = exp(2*alpha*old_Phi)
Lambda_expr(r, th) = 1 + (1+ alpha**2) / 4 * B**2 * (r*sin(th))**2 * old_Psi.expr()

In [32]:
show(LatexExpr(r'\Lambda = '), Lambda_expr(r, th))
show(LatexExpr(r'A_v(r) = '), Av_expr(r), LatexExpr(r', \quad\quad    E_v(r,\theta) = '), Ev_expr(r,th))
show(LatexExpr(r"\phi_{0} = "), old_Phi.expr())
show(LatexExpr(r"\Psi_{0} = e^{2\alpha\phi_{0}} = "), old_Psi.expr())

\Lambda =  13/48*B^2*r^2*e^(1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))*sin(th)^2 + 1

A_v(r) =  -2*m/r + 1 , \quad\quad    E_v(r,\theta) =  e^(1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2)

\phi_{0} =  m*xi/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1))

\Psi_{0} = e^{2\alpha\phi_{0}} =  e^(1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))

In [33]:
Phi = M.scalar_field({XY: - alpha / (1+alpha**2) * log(Lambda(r,th)) + old_Phi.expr()}, name='Phi', latex_name=r'\Phi')
Psi = M.scalar_field({XY: old_Psi.expr()/ (Lambda(r,th))**(2*alpha**2/(1+alpha**2))}, name=r'\Psi')

Phi_elec = M.scalar_field({XY: -Phi.expr()}, name=r'\phi') 
Psi_elec = M.scalar_field({XY: 1/Psi.expr()}, name=r'\Psi')

varth_elec = M.scalar_field({XY:Psi_elec.expr()**(-1)}, name=r'\vartheta') # varth_elec=1/Psi_elec=e^{-2*\alpha*\phi}

Omega_expr(r,th) = Psi_elec.expr()

show(LatexExpr(r"\Psi' = e^{2\alpha\phi'} = "), Psi_elec.expr())
show(LatexExpr(r"\varphi' = "), Phi_elec.expr())
show(LatexExpr(r"\Omega = e^{2\alpha\phi'} = \Psi' = "), Omega_expr(r,th))
show(LatexExpr(r"\vartheta = e^{-2\alpha\phi'} = \frac{1}{\Psi'} ="), varth_elec.expr())

\Psi' = e^{2\alpha\phi'} =  Lambda(r, th)^(2/13)*e^(-1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))

\varphi' =  -m*xi/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1)) + 2/13*sqrt(3)*log(Lambda(r, th))

\Omega = e^{2\alpha\phi'} = \Psi' =  Lambda(r, th)^(2/13)*e^(-1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))

\vartheta = e^{-2\alpha\phi'} = \frac{1}{\Psi'} = e^(1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))/Lambda(r, th)^(2/13)

In [34]:
g_nod = g.copy()
g_nod = subs_func(g_nod)
nod_ratio = (g_nod[3,3]/(sin(th)**2*g_nod[2,2])).factor()
show(LatexExpr(r"\lim_{\theta\rightarrow 0} \frac{g_{\phi\phi}}{\sin^{2}\theta g_{\theta\theta}} = " + latex(limit(nod_ratio.expr(), th=0).canonicalize_radical())))

Omega substituted
Lambda substituted
Av substituted
Ev substituted
Full Substitution : Done


\lim_{\theta\rightarrow 0} \frac{g_{\phi\phi}}{\sin^{2}\theta g_{\theta\theta}} = 1

In [ ]:
R = g.ricci_scalar() #Ricci scalar

In [ ]:
Ric = g.ricci() #Ricci tensor

In [ ]:
G = Ric-R*g/2 #Einstein tensor

# IV. Definition of the EM tensor

In [ ]:
F_EM = diff(pot_vec)
F_EM.display()

As Sage is unable to verify the field equations using this fully explicit expression of the electromagnetic tensor, we revert to a more general formulation. 

In [ ]:
F = M.diff_form(2, 'F')
F[0,1] = 24*abs(Ev(r,th))*exp(-sqrt(3)*sqrt(2)*m*xi/(6*sqrt(m^2*cos(th)^2 - 2*m*r + r^2)))*diff(Lambda(r,th), th) / (13*B*r^2*Ev(r,th)*sin(th))
F[0,2] = -24*Av(r)*abs(Ev(r,th))*exp(-sqrt(3)*sqrt(2)*m*xi/(6*sqrt(m^2*cos(th)^2 - 2*m*r + r^2)))*diff(Lambda(r,th), r) / (13*B*Ev(r,th)*sin(th))

We nevertheless confirm that, after substituting all expressions, the same electromagnetic tensor is recovered.

In [ ]:
F_test = F.copy()
F_d = subs_func(F_test)
(F_EM-F_d).is_zero()

In [ ]:
E_elec = M.tensor_field(1,0)
for i in [1, 2, 3]:  
    E_elec[i] = F_EM[0, i].expr().canonicalize_radical() 
show(LatexExpr(r'E^{\mu} = '),E_elec[:])

# V. Field equations

## V. 1. The scalar-field form $-\frac{\mathcal{L}_m}{R} = \vartheta$

In [ ]:
Lm = -F['_ij']*F.up(g)['^ij']/2

In [ ]:
eq = Lm/R + varth_elec
eq = subs_func(eq)

In [ ]:
var_dict = {"eq": eq}
save(var_dict, "ER_Penney_Melvin_dict.sobj")

In [ ]:
# after section IV, you can directly jump to this cell and load 'eq' that can be directly tested in the next cell
#globals().update(load("ER_Penney_Melvin_dict.sobj"))

In [ ]:
latex_str = r'\frac{\mathcal{L}_m}{R} + \vartheta = ' 
show(LatexExpr(latex_str), numerator(eq.factor()).simplify_full())

## V. 2. Electromagnetic field equation

In [ ]:
eq_MxW = nab(varth_elec * F.up(g))['_i^ij']
eq_MxW = subs_func(eq_MxW)
eq_MxW.apply_map(lambda f: f.simplify_trig())

In [ ]:
latex_str = r'\triangledown_{\sigma}\left(\frac{\mathcal{L}_m}{R}F^{\mu\sigma}\right) = ' 
show(LatexExpr(latex_str), eq_MxW[:])

## V. 3. The metric field equation

Let's define the stress-energy tensor $T_{\mu\nu}=2\left(F_{\rho\mu}F^{\rho}_{\hspace{0.2cm}\nu}-\frac{1}{4}g_{\mu\nu}F^{2}\right)$ 

In [ ]:
T = 2*(F.up(g,1)['_i^j']*F['_kj'] + g*Lm/2)

and verify that its trace is zero

In [ ]:
T.up(g,1).trace(0,1).display()

In [ ]:
S = (nab(nab(varth_elec**2)) - g*(varth_elec**2).dalembertian()) / varth_elec**2

We deduce the field equation

In [ ]:
eq_m = G + R / Lm * T - S

In [ ]:
eq_m_test = eq_m.copy()
for i in range(4):
    for j in range(4):
        eq_m_test[i,j] = eq_m_test[i,j].expr().substitute_function(Omega, Omega_expr).substitute_function(Lambda, Lambda_expr).substitute_function(Av, Av_expr).substitute_function(Ev, Ev_expr)

var_dict = {"test_00": eq_m_test[0,0],"test_11": eq_m_test[1,1],"test_22": eq_m_test[2,2], "test_33": eq_m_test[3,3], "test_12": eq_m_test[1,2], "test_21": eq_m_test[2,1]}
save(var_dict, "ER_Penney_Melvin_test_dict.sobj")

In [ ]:
# after section IV, you can directly jump to this cell to load all the eq_ij components that can be quickly tested
# globals().update(load("ER_Penney_Melvin_test_dict.sobj"))

All the components $eq_m[i,j]$ of this tensor equation cannot be easily simplified to assess whether they vanish. Their explicit expressions involve long combinations of algebraic terms and exponential functions, whose direct simplification in SageMath is impractical due to their complexity. To demonstrate that these components are identically zero, we show that the numerator of $eq_m[i,j]$ can be written as:

$eq_m[i,j] = A + \sum_n C_n\,\exp(f_n)$

where $A$ and $C_n$ are algebraic expression and the set $f_n$ contains no redundant terms. Since exponential functions never vanish, the algebraic contribution $A$ must be identically zero for the full expression to vanish. See the appendix for more details.

### Component $eq_{00}$

In [ ]:
test_00 = eq_m_test[0,0].expr()

In [ ]:
'''The expression'''
expr = test_00
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

In [ ]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

In [ ]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

### Component $eq_{11}$

In [ ]:
test_11 = eq_m_test[1,1].expr()

In [ ]:
'''The expression'''
expr = test_11
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

In [ ]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

In [ ]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

### Component $eq_{22}$

In [ ]:
test_22 = eq_m_test[2,2].expr()

In [ ]:
'''The expression'''
expr = test_22
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

In [ ]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

In [ ]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

### Component $eq_{33}$

In [ ]:
test_33 = eq_m_test[3,3].expr()

In [ ]:
'''The expression'''
expr = test_33
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

In [ ]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

### Component $eq_{12}$

In [ ]:
test_12 = eq_m_test[1,2].expr()

In [ ]:
'''The expression'''
expr = test_12
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

In [ ]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

In [ ]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

In [ ]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

### Component $eq_{21}$

In [ ]:
test_21 = eq_m_test[2,1].expr()

In [ ]:
'''The expression'''
expr = test_21
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

In [ ]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

In [ ]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

In [ ]:
eq_result = eq_m.copy()

for i in range(4):
    for j in range(4):
        if [i,j] in [[0,0],[1,1],[2,2],[3,3],[1,2],[2,1]]:
            eq_result[i,j]=0

In [ ]:
latex_str = r'G_{\mu\nu} + \frac{R}{\mathcal{L}_{m}}T_{\mu\nu}-\frac{1}{\vartheta^2}\left[\nabla_{\mu}\nabla_{\nu} - g_{\mu\nu}\square\right]\vartheta^{2} = ' 
show(LatexExpr(latex_str), eq_result[:])

# WE CONCLUDE THAT EMBEDDING THE PENNEY SOLUTION INTO AN EXTERNAL ELECTRIC FIELD GENERATES NEW SOLUTIONS IN ER